In [ ]:
import polars as pl


pl.set_random_seed(42)
annot_path = '../references/Homo_sapiens/gencode.v49.annotation.gtf.gz'
genome_fasta_path = '../references/Homo_sapiens/GRCh38.p14.genome.bgzip.fa.gz'
gene_count=20
only_protein_coding=False
transcript_count_min=4
transcript_count_max=20
gene_length_min=4000
gene_length_max=40000

GTF_COLUMNS = ['chr', 'source', 'feature', 'start', 'end', 'score', 'strand', 'frame', 'attr']

annot_lf = pl.scan_csv(
    annot_path, separator='\t', comment_prefix='#', has_header=False,
    new_columns=GTF_COLUMNS
)

# count transcripts per gene, and filter genes with too many or too few transcripts
transcript_lf = (
    annot_lf
    .select(['feature', 'attr'])
    .filter(pl.col('feature') == 'transcript')
    .filter(pl.col('attr').str.contains('gene_type "protein_coding";') if only_protein_coding else pl.lit(True))
    .with_columns(pl.col('attr').str.extract(r'gene_id "([^"]+)";').alias('gene_id'))
    .select(
        pl.col('gene_id').unique(maintain_order=True),
        pl.col('gene_id').unique_counts().alias('transcript_count')
    )
    .filter(pl.col('transcript_count').is_between(transcript_count_min, transcript_count_max))
)
filtered_gene_pool = transcript_lf.collect()
filtered_gene_ids = filtered_gene_pool['gene_id'].to_list()

# calc gene length, and filter genes with too long or too short length
gene_lf = (
    annot_lf
    .filter(pl.col('feature') == 'gene')
    .with_columns(pl.col('attr').str.extract(r'gene_id "([^"]+)";').alias('gene_id'))
    .filter(pl.col('gene_id').is_in(filtered_gene_ids))
    .with_columns((pl.col('end') - pl.col('start') + 1).alias('gene_length'))
    .filter(pl.col('gene_length').is_between(gene_length_min, gene_length_max))
)

# pick random genes
sample_gene_ids = gene_lf.collect().sample(gene_count)['gene_id'].to_list()
sample_lf = (
    annot_lf
    .with_columns(pl.col('attr').str.extract(r'gene_id "([^"]+)";').alias('gene_id'))
    .filter(pl.col('gene_id').is_in(sample_gene_ids))
    .select(GTF_COLUMNS + ['gene_id'])
)

sample_annot_df = sample_lf.collect()
sample_annot_df

In [ ]:
from typing import Literal
from pyfaidx import Fasta


def calc_gap_length(gene_df: pl.DataFrame, current_pos: int, chr_size: int, compared: Literal['next', 'prev']='next', max_length=100000) -> int:
    """Tool function, limit max gap length to 100kb."""
    if compared not in ['next', 'prev']:
        raise ValueError("compared must be 'next' or 'prev'")
    if gene_df.is_empty():
        print(f"Warning: no gene found.")
        if compared == 'next':
            return min(chr_size - current_pos, max_length)
        else:
            return min(current_pos - 1, max_length)
    elif gene_df.shape[0] == 1:
        if compared == 'next':
            return min(gene_df[0, 'start'] - current_pos - 1, max_length)
        else:
            return min(current_pos - gene_df[0, 'end'] - 1, max_length)
    else:
        raise ValueError("gene_df should contain one gene")
    

def write_fasta(out_file_path: str, sequences: dict[str, str], width: int=80):
    """Write sequences to FASTA file with specified width."""
    with open(out_file_path, 'w') as f:
        for seq_id, seq in sequences.items():
            f.write(f'>{seq_id}\n')
            for i in range(0, len(seq), width):
                f.write(seq[i:i+width] + '\n')
        

sample_gene_df = (
    sample_annot_df
    .filter(pl.col('feature') == 'gene')
    .with_columns((pl.col('end') - pl.col('start') + 1).alias('gene_length'))
)

sample_chrs = sample_gene_df['chr'].unique().to_list()
all_needed_genes_lf = (
    annot_lf
    .filter(pl.col('feature') == 'gene', pl.col('chr').is_in(sample_chrs))
    .select(pl.all().sort_by('start').over('chr'))
)
all_needed_genes_df = all_needed_genes_lf.collect()

total_length = 0
coord_shift_records = {} # genome coordinate start (bp, 1-based) for each gene after shifting
genome_fasta = Fasta(genome_fasta_path) # slicing use 0-based
mock_genome_sequence = ''

for idx, row in enumerate(sample_gene_df.iter_rows(named=True)):
    chr_size = len(genome_fasta[row['chr']])
    if idx == 0:
        prev_gene_df = (
            all_needed_genes_df
            .filter(pl.col('chr') == row['chr'], pl.col('end') < row['start'])
            .tail(1)
        )
        prev_gap_length = calc_gap_length(prev_gene_df, row['start'], chr_size, 'prev')
        mock_genome_sequence += genome_fasta[row['chr']][(row['start'] - prev_gap_length - 1):(row['start'] - 1)].seq.upper()
        total_length += prev_gap_length
    
    coord_shift_records[row['gene_id']] = row['start'] - total_length - 1
    # get next gene coord to calculate intergenic region length
    next_gene_df = (
        all_needed_genes_df
        .filter(pl.col('chr') == row['chr'], pl.col('start') > row['end'])
        .head(1)
    )
    next_gap_length = calc_gap_length(next_gene_df, row['end'], chr_size, 'next')
    mock_genome_sequence += genome_fasta[row['chr']][(row['start'] - 1):(row['end'] + next_gap_length)].seq.upper()
    total_length += (row['gene_length'] + next_gap_length)


mock_annot_df = (
    sample_annot_df
    .with_columns(
        chr = pl.lit('chr1'),
        source = pl.lit('TEST'),
        shift_value = pl.col('gene_id').replace_strict(coord_shift_records)
    )
    .with_columns(
        start = pl.col('start') - pl.col('shift_value'),
        end = pl.col('end') - pl.col('shift_value')
    )
    .select(GTF_COLUMNS)
)

out_path = '../test/data'
mock_annot_df.write_csv(f'{out_path}/annotation.gtf', include_header=False, separator='\t', quote_style='never')
write_fasta(f'{out_path}/genome.fa', {'chr1': mock_genome_sequence})



In [23]:
import polars as pl
import numpy as np

np.random.seed(42)

splice_annot_path = '../test/data/annotation.mock_spliced.gtf'
GTF_COLUMNS = ['chr', 'source', 'feature', 'start', 'end', 'score', 'strand', 'frame', 'attr']
annot_lf = pl.scan_csv(
    splice_annot_path, separator='\t', comment_prefix='#', has_header=False,
    new_columns=GTF_COLUMNS
)
transcript_count_lf = (
    annot_lf
    .filter(pl.col('feature') == 'transcript')
    .select(
        gene_id=pl.col('attr').str.extract(r'gene_id "([^"]+)"'),
        transcript_id=pl.col('attr').str.extract(r'transcript_id "([^"]+)"')
    )
    .unique(subset='transcript_id', maintain_order=True)
    .group_by('gene_id', maintain_order=True)
    .agg(
        transcript_count=pl.len(),
        transcript_ids=pl.col('transcript_id')
    )
)


transcript_count_df = transcript_count_lf.collect()
abundance_df = transcript_count_df.with_columns(
    gene_abundance = pl.lit((np.random.normal(100, 20, size=transcript_count_df.height))).cast(pl.UInt32)
)

abundance_df = abundance_df.filter(pl.col('gene_id') == 'gene_1')

replicate_count = 5
sample_labels = [f'control_{i}' for i in range(1, replicate_count + 1)] + [f'experiment_{i}' for i in range(1, replicate_count + 1)]
control_libsize = np.random.normal(10, 1.5, replicate_count)
experiment_libsize = np.random.normal(10, 1.5, replicate_count)

output_rows = [
    'gene_id\ttranscript_id\t' + '\t'.join(sample_labels)
]
for row in abundance_df.iter_rows(named=True):
    transcript_ids = row['transcript_ids']
    # half for experiment, half for control. assume transcripts are nearly equally expressed
    control_weights = [1 if idx.endswith('.N') else 20 for idx in transcript_ids ]
    experiment_weights = [20] * len(transcript_ids) # assume novel splicing over-expressed in experiment

    transcript_usages = np.random.dirichlet(control_weights + experiment_weights) * row['gene_abundance'] * 2

    control_matrix = np.outer(transcript_usages[:len(transcript_ids)], control_libsize)
    experiment_matrix = np.outer(transcript_usages[len(transcript_ids):], experiment_libsize)

    all_matrix = np.hstack([control_matrix, experiment_matrix])
    # add random noise
    all_matrix *= np.random.normal(1, 0.1, all_matrix.shape)

    for i, transcript_id in enumerate(transcript_ids):
        output_rows.append(
            f'{row['gene_id']}\t{transcript_id}\t' + '\t'.join(all_matrix[i].astype(str))
        )
    

print('\n'.join(output_rows))




gene_id	transcript_id	control_1	control_2	control_3	control_4	control_5	experiment_1	experiment_2	experiment_3	experiment_4	experiment_5
gene_1	transcript_1.1	199.59998355025553	133.0107322964826	147.11364715582604	122.08207576295503	151.47737625838252	175.5072696361357	147.23291459480507	170.3509367190573	145.25112242735116	187.47818857623437
gene_1	transcript_1.2	353.66007435624806	244.87502170788963	283.76255168961404	207.99641536321425	219.32541252740486	220.50160969097593	199.82448011943274	220.33225283165953	220.2697846082698	147.72975190824914
gene_1	transcript_1.3	272.88903878703854	201.45507528761624	202.5676613088949	164.03082482570994	152.1058907583936	135.5087914328237	116.78202885025102	165.24165034582768	117.57927681762504	119.7853867262789
gene_1	transcript_1.4	149.8860350138111	136.42354408483172	134.96997684762684	96.32852713535046	124.89782080135927	121.19792261473	107.14594429440362	115.96576875700327	103.91066048867958	108.47525711450382
gene_1	transcript_1.5	131.15

In [8]:
import polars as pl
from pyfaidx import Fasta


def extract_sequence(chr: str, strand: Literal['+','-'], exons: list[tuple[int, int]], genome_fasta: Fasta) -> str:
    """Return sequence of transcript. Exons are 1-based (from gtf)."""
    for i in range(len(exons) - 1):
        assert exons[i][0] < exons[i][1], f"Exon {i+1} start should be less than end, {exons}"
        assert exons[i][1] <= exons[i + 1][0], f"Exon {i+1} end should not overlap with next exon, {exons}"

    seq_list = []
    for start, end in exons:
        start -= 1 # convert to 0-based for pyfaidx
        if strand == '+':
            seq_list.append(genome_fasta[chr][start:end].seq.upper())
        else:
            seq_list.append(genome_fasta[chr][start:end].reverse.complement.seq.upper())
    
    if strand == '+':
        return ''.join(seq_list)
    else:
        return ''.join(seq_list[::-1])

GENOME_FASTA = Fasta('../test/data/genome.fa')

GTF_COLUMNS = ['chr', 'source', 'feature', 'start', 'end', 'score', 'strand', 'frame', 'attr']
sample_annot_lf = pl.scan_csv(
    '../test/data/annotation.mock_spliced.gtf', separator='\t', comment_prefix='#', has_header=False,
    new_columns=GTF_COLUMNS
)
exons_lf = (
    sample_annot_lf
    .filter(pl.col('feature') == 'exon')
    .with_columns(
        transcript_id = pl.col('attr').str.extract(r'transcript_id "([^"]+)"'),
        coord = pl.concat_list('start', 'end')
    )
    .group_by('transcript_id', maintain_order=True)
    .agg(
        exons = pl.col('coord'),
        strand = pl.first('strand'),
        chr = pl.first('chr')
    )
    .with_columns(pl.col('exons').map_elements(lambda e: sorted(e, key=lambda x: x[0])))
    .select(
        ['transcript_id', 'chr', 'strand', 'exons'],
        sequence = pl.struct(
            'chr', 'strand', 'exons'
        ).map_elements(
            lambda row: extract_sequence(row['chr'], row['strand'], row['exons'], GENOME_FASTA),
            pl.String
        )
    )
    .with_columns(
        length = pl.col('sequence').str.len_chars()
    )
)
exons_lf.collect()


transcript_id,chr,strand,exons,sequence,length
str,str,str,list[list[i64]],str,u32
"""transcript_1.1""","""chr1""","""-""","[[1677, 2944], [4036, 4092], … [34134, 34404]]","""AGACCAGCTCCAGGCGCTGGGGCTTTCTCA…",2495
"""transcript_1.2""","""chr1""","""-""","[[1677, 2944], [4036, 4092], … [34134, 34421]]","""GGCTTGTGGGAACTGTCAGACCAGCTCCAG…",1918
"""transcript_1.3""","""chr1""","""-""","[[2700, 2944], [4036, 4092], … [34134, 34277]]","""GCTGACTGGGGACTGCTGAGCCCGTGCACG…",1360
"""transcript_1.4""","""chr1""","""-""","[[2788, 2944], [4036, 4092], … [34134, 34217]]","""CCTGGCCCCTCTGTCCTGTGGAAATGCTGG…",921
"""transcript_1.5""","""chr1""","""-""","[[2788, 2944], [4036, 4236], … [34134, 34217]]","""CCTGGCCCCTCTGTCCTGTGGAAATGCTGG…",1341
…,…,…,…,…,…
"""transcript_20.9""","""chr1""","""-""","[[651083, 651578], [652098, 652238], … [657408, 657554]]","""GTCACGGATCGGCAGCGGGACTGAGCCGCG…",1473
"""transcript_20.10""","""chr1""","""-""","[[651088, 651513], [654067, 654546], [656718, 656932]]","""CCTTTCGGTCCAGGCGGCGGCAGGGCTGAG…",1121
"""transcript_20.11""","""chr1""","""-""","[[651334, 651578], [652098, 652238], … [657256, 657506]]","""CAGGCTTCGCGCGGGGCCCCCCGGGTGCAC…",1326


In [9]:
read_length = 100

transcript_usage_df = pl.read_csv('../test/data/mock_transcripts_usage.tsv', separator='\t')

sample_cols = [col for col in transcript_usage_df.columns if col.startswith('sample')]
transcript_usage_df.join(exons_lf.collect(), on='transcript_id', how='inner').with_columns(
            **{sample: (pl.col(sample) * pl.col('length') / read_length).cast(pl.UInt32) for sample in sample_cols}
        )

gene_id,transcript_id,sample.control_1,sample.control_2,sample.control_3,sample.control_4,sample.control_5,sample.experiment_1,sample.experiment_2,sample.experiment_3,sample.experiment_4,sample.experiment_5,chr,strand,exons,sequence,length
str,str,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,str,str,list[list[i64]],str,u32
"""gene_1""","""transcript_1.1""",4965,3318,3667,3043,3767,4366,3667,4241,3617,4665,"""chr1""","""-""","[[1677, 2944], [4036, 4092], … [34134, 34404]]","""AGACCAGCTCCAGGCGCTGGGGCTTTCTCA…",2495
"""gene_1""","""transcript_1.2""",6770,4679,5427,3970,4200,4219,3816,4219,4219,2819,"""chr1""","""-""","[[1677, 2944], [4036, 4092], … [34134, 34421]]","""GGCTTGTGGGAACTGTCAGACCAGCTCCAG…",1918
"""gene_1""","""transcript_1.3""",3699,2733,2747,2230,2067,1836,1577,2244,1591,1618,"""chr1""","""-""","[[2700, 2944], [4036, 4092], … [34134, 34277]]","""GCTGACTGGGGACTGCTGAGCCCGTGCACG…",1360
"""gene_1""","""transcript_1.4""",1372,1252,1234,884,1142,1114,985,1059,948,994,"""chr1""","""-""","[[2788, 2944], [4036, 4092], … [34134, 34217]]","""CCTGGCCCCTCTGTCCTGTGGAAATGCTGG…",921
"""gene_1""","""transcript_1.5""",1756,1676,1743,1327,1501,1461,1327,1703,1394,1568,"""chr1""","""-""","[[2788, 2944], [4036, 4236], … [34134, 34217]]","""CCTGGCCCCTCTGTCCTGTGGAAATGCTGG…",1341
…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…
"""gene_20""","""transcript_20.9""",780,500,736,603,662,736,662,854,780,824,"""chr1""","""-""","[[651083, 651578], [652098, 652238], … [657408, 657554]]","""GTCACGGATCGGCAGCGGGACTGAGCCGCG…",1473
"""gene_20""","""transcript_20.10""",885,762,829,706,773,605,459,661,515,482,"""chr1""","""-""","[[651088, 651513], [654067, 654546], [656718, 656932]]","""CCTTTCGGTCCAGGCGGCGGCAGGGCTGAG…",1121
"""gene_20""","""transcript_20.11""",888,663,623,477,609,596,464,609,596,596,"""chr1""","""-""","[[651334, 651578], [652098, 652238], … [657256, 657506]]","""CAGGCTTCGCGCGGGGCCCCCCGGGTGCAC…",1326
